#  추천 시스템 - ETF 데이터 수집 및 전처리 


---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob

from pprint import pprint
import json

## ETF 데이터 수집

- CSV 다운로드
- http://data.krx.co.kr/contents/MDC/MDI/mdiLoader/index.cmd?menuId=MDC020103010901

In [3]:
import pandas as pd
import numpy as np

etf_list = pd.read_csv('data/etf_list.csv', encoding='cp949')
etf_list.head()

,종목코드,종목명,상장일,분류체계,운용사,수익률(최근 1년),기초지수,추적오차,순자산총액,괴리율,변동성,복제방법,총보수,과세유형
0,466400,1Q 25-08 회사채(A+이상)액티브,2023/09/19,채권-회사채-단기,하나자산운용,4.52,KIS 2025-08만기형 크레딧 A+이상 지수(총수익),0.11,111916276404,0.03,매우낮음,실물(액티브),0.10,배당소득세(보유기간과세)
1,491610,1Q CD금리액티브(합성),2024/09/24,기타,하나자산운용,0.00,KIS 하나 CD금리 총수익지수,0.05,316206006696,0.02,매우낮음,합성(액티브),0.02,배당소득세(보유기간과세)
2,451060,1Q K200액티브,2023/01/31,주식-시장대표,하나자산운용,-3.66,코스피 200,0.77,99754348820,-0.01,높음,실물(액티브),0.18,배당소득세(보유기간과세)
3,463290,1Q 단기금융채액티브,2023/08/03,채권-혼합-단기,하나자산운용,4.01,MK 머니마켓 지수(총수익),0.05,252717462257,0.00,매우낮음,실물(액티브),0.08,배당소득세(보유기간과세)
4,479080,1Q 머니마켓액티브,2024/04/02,채권-혼합-단기,하나자산운용,0.00,KIS-하나 MMF 지수(총수익),0.06,308255065986,-0.01,매우낮음,실물(액티브),0.05,배당소득세(보유기간과세)


In [4]:
etf_list.tail()

,종목코드,종목명,상장일,분류체계,운용사,수익률(최근 1년),기초지수,추적오차,순자산총액,괴리율,변동성,복제방법,총보수,과세유형
925,429870,히어로즈 리츠이지스액티브,2022/05/24,부동산-리츠,키움투자자산운용,-4.51,iSelect 리츠 지수,3.07,3763127456,-0.98,낮음,실물(액티브),0.300,배당소득세(분리과세부동산ETF)
926,476450,히어로즈 머니마켓액티브,2024/02/29,채권-혼합-단기,키움투자자산운용,0.00,KIS-키움 MMF지수(총수익지수),0.06,327428123675,0.00,매우낮음,실물(액티브),0.050,배당소득세(보유기간과세)
927,460270,히어로즈 미국달러SOFR금리액티브(합성),2023/06/20,기타,키움투자자산운용,16.59,Solactive SOFR Daily Total Return Index,0.09,9885129694,-0.07,매우낮음,합성(액티브),0.050,배당소득세(보유기간과세)
928,459790,히어로즈 미국성장기업30액티브,2023/06/27,주식-시장대표,키움투자자산운용,44.70,MSCI USA Index(Price Return),6.44,13371801852,0.31,높음,실물(액티브),0.760,배당소득세(보유기간과세)
929,454780,히어로즈 종합채권(AA-이상)액티브,2023/04/11,채권-혼합-중기,키움투자자산운용,7.56,KIS 종합 채권시장지수(AA-이상)(총수익지수),0.24,338847308716,0.26,매우낮음,실물(액티브),0.025,배당소득세(보유기간과세)


In [5]:
etf_list.shape

(930, 14)

In [6]:
etf_list[etf_list['종목코드'] == '466400']

,종목코드,종목명,상장일,분류체계,운용사,수익률(최근 1년),기초지수,추적오차,순자산총액,괴리율,변동성,복제방법,총보수,과세유형
0,466400,1Q 25-08 회사채(A+이상)액티브,2023/09/19,채권-회사채-단기,하나자산운용,4.52,KIS 2025-08만기형 크레딧 A+이상 지수(총수익),0.11,111916276404,0.03,매우낮음,실물(액티브),0.1,배당소득세(보유기간과세)


In [7]:
strIsurCd = '45979'
etf_url = f"https://kind.krx.co.kr/disclosure/etfisudetail.do?method=searchEtfIsuSummary&strIsurCd={strIsurCd}"

etf_url

'https://kind.krx.co.kr/disclosure/etfisudetail.do?method=searchEtfIsuSummary&strIsurCd=45979'

## Crawl4AI

- LLM, AI 에이전트, 데이터 파이프라인을 위한 **고성능 웹 크롤링** 솔루션 제공

- **오픈소스** 기반으로 실시간 성능과 유연한 구현이 가능

- https://github.com/unclecode/crawl4ai?tab=readme-ov-file

- 설치 방법:
    ```bash
    # 기본 설치 - 코어 라이브러리만 설치
    pip install crawl4ai
    uv add crawl4ai

    # 초기 설정 - Playwright 브라우저 설치 및 환경 점검
    (uv run) crawl4ai-setup

    # 진단 도구 - 시스템 호환성 확인
    (uv run) crawl4ai-doctor
    ```

### 1) Jupyter 환경 설정

- `nest_asyncio`는 Jupyter에서 이미 실행 중인 이벤트 루프 위에 중첩된 이벤트 루프를 실행할 수 있게 해주는 패키지

- Jupyter Notebook 에서 asyncio 를 사용할 때, 이 코드를 실행하면 에러가 발생하지 않음 

In [8]:
# Jupyter Notebook 에서 asyncio 를 사용할 때, 이 코드를 실행하면 에러가 발생하지 않음 (MacOS, Linux 환경)
# import nest_asyncio
# nest_asyncio.apply()

In [9]:
# # Jupyter Notebook 에서 asyncio 를 사용할 때, 이 코드를 실행하면 에러가 발생하지 않음 (Windows용)
import asyncio
import sys

# 이벤트 루프 정책 변경 (Windows용)
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
# 새 이벤트 루프 생성 및 설정
loop = asyncio.new_event_loop()
asyncio.set_event_loop(loop)

# nest_asyncio 적용
import nest_asyncio
nest_asyncio.apply(loop)

### 2) **기본 크롤링** 

- `BrowserConfig`와 `CrawlerRunConfig` 클래스 기본값 설정을 사용

In [10]:
import asyncio
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig

async def main():
    """ 
    크롤링 실행 함수
    """
    browser_config = BrowserConfig()  # 브라우저 설정
    run_config = CrawlerRunConfig()   # 크롤러 실행 설정

    async with AsyncWebCrawler(config=browser_config) as crawler:   # 크롤러 객체 생성
        result = await crawler.arun(
            url=etf_url,
            config=run_config   # 크롤러 실행 설정
        )
        
    return result


# 크롤링 실행 및 결과 출력
# mac
#result = asyncio.run(main())  # asyncio.run() 사용

#window
loop = asyncio.new_event_loop()
asyncio.set_event_loop(loop)
result = loop.run_until_complete(main())  # 새 이벤트 루프에서 실행
#print(result.markdown)  # 크롤링 결과 (정제된 markdown 출력)

c:\fun\002_etfbot\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


[INIT].... → Crawl4AI 0.8.0 

[FETCH]... ↓ https://kind.krx.co.kr/disclosure/etfisudetail.do?method=searchEtfIsuSummary&strIsurCd=45979         |
✓ | ⏱: 0.67s 

[SCRAPE].. ◆ https://kind.krx.co.kr/disclosure/etfisudetail.do?method=searchEtfIsuSummary&strIsurCd=45979         |
✓ | ⏱: 0.01s 

[COMPLETE] ● https://kind.krx.co.kr/disclosure/etfisudetail.do?method=searchEtfIsuSummary&strIsurCd=45979         |
✓ | ⏱: 0.69s 

### 3) **CrawlResult 응답 객체** 

- 크롤링된 URL, HTML, 성공 여부 등 기본 정보 포함

- **cleaned_html**와 **markdown** 필드로 정제된 데이터 접근 가능

- **extracted_content**를 통해 구조화된 형태의 추출 데이터 확인

In [11]:
result.model_dump()

{'url': 'https://kind.krx.co.kr/disclosure/etfisudetail.do?method=searchEtfIsuSummary&strIsurCd=45979',
 'html': '<!DOCTYPE html><html lang="ko"><head>\n<meta http-equiv="X-UA-Compatible" content="IE=edge">\n<meta http-equiv="content-type" content="text/html; charset=utf-8">\n<meta http-equiv="content-language" content="kr">\n<meta http-equiv="content-style-type" content="text/css">\n<meta http-equiv="pragma" content="no-cache">\n<meta http-equiv="cache-control" content="no-cache">\n<meta http-equiv="imagetoolbar" content="no">\n<meta name="copyright" content="(c)2012">\n<meta name="description" content="대한민국 대표 기업공시채널 KIND">\n<meta name="keywords" content="대한민국 대표 기업공시채널 KIND">\n<title>대한민국 대표 기업공시채널 KIND</title>\n<link rel="stylesheet" type="text/css" href="../js/jquery/themes-base/jquery-ui.min.css" media="all">\n<link rel="stylesheet" type="text/css" href="../css/default_new.css" media="all">\n<link rel="stylesheet" type="text/css" href="../css/common.css?version=20240531" media="a

In [12]:
# 크롤링 성공 여부 확인
result.success

True

In [13]:
# HTML 접근
print(result.html[:100])

<!DOCTYPE html><html lang="ko"><head>
<meta http-equiv="X-UA-Compatible" content="IE=edge">
<meta ht


In [14]:
# 정제된 HTML 출력
print(result.cleaned_html[:100])

<html>
<head>
<title>대한민국 대표 기업공시채널 KIND</title>
<!--[if lt IE 9]>
<script type="text/javascript" sr


In [15]:
# 마크다운 변환 결과 (정제된)
print(result.markdown)

# 키움 KIWOOM 미국성장기업30액티브증권상장지수투자신탁[주식]
  * [상품개요(ETF)](https://kind.krx.co.kr/disclosure/etfisudetail.do?method=searchEtfIsuSummary&strIsurCd=45979 "상품개요\(ETF\)")


## 상품 개요
목록 한글명 |  KIWOOM 미국성장기업30액티브  | 영문명 |  KIWOOM US Growth 30 Active   
---|---|---|---  
표준코드 |  KR7459790002  | 종목코드 |  459790   
상장일 |  2023-06-27  | 펀드형태 |  수익증권형   
기초지수명 |  MSCI USA Index(Price Return)  | 추적배수 |  일반 (1)   
자산운용사 |  키움투자자산운용(주)  | 지정참가회사(AP) |  키움증권, 삼성증권, DB금융투자   
총보수(%) |  0.76  | 회계기간 |  매년 1월 1일부터 12월 31일까지. 단, 최초 회계기간은 최초설정일로부터 당해연도 12월 31일까지   
과세유형 |  배당소득세(보유기간과세)  | 분배금 지급일 |  1월, 4월, 7월, 10월 마지막 영업일 및 회계기간 종료일(다만, 회계기간 종료일이 영업일이 아닌 경우 그 직전 영업일)   
홈페이지 |  <http://www.kosef.co.kr>  
## 유형 정보
목록 기초 시장 |  (해외)   
(북미)   
(미국)  | 기초 자산 |  (주식)   
(시장대표) / (-)   
(-) / (-)   
---|---|---|---  
## 상품 설명
목록 기본 정보 |  ```
1. 이 투자신탁은 해외 주식을 주된 투자대상으로 하며, MSCI에서 산출 및 발표하는 “MSCI USA Index(Price Return)(원화환산)”를 비교지수로 하여 1좌당 순자산가치의 변동률이 비교지수의 변동률을 초과하도록 운용하는 것을 목적으로 합니다.  

2. 이 투자신탁은 액티브상장지수펀드로서 비교지

In [16]:
# 구조화된 데이터 (Optional)
if result.extracted_content:
    data = json.loads(result.extracted_content)
    print(data)

### 4) **BrowserConfig 설정** 

- **BrowserConfig**로 브라우저 타입, 헤드리스 모드, 뷰포트 크기 등 **기본 설정** 관리

- **프록시 설정**과 **디버깅 옵션**을 통한 고급 크롤링 환경 구성

- **AsyncWebCrawler** 클래스와 연동하여 비동기 크롤링 실행

In [18]:
import asyncio
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig

# 기본 설정
base_config = BrowserConfig(
    browser_type="chromium",  # 브라우저 엔진 선택
    viewport_width=1080,     # 뷰포트 크기
    viewport_height=600,
    text_mode=True,          # 텍스트 모드 (이미지 비활성화)
    use_persistent_context=True,  # 세션을 유지
    headless=True,           # 헤드리스 모드 실행
)

# 디버깅용 설정
debug_config = base_config.clone(
    headless=False,  # 헤드리스 모드 비활성화
    verbose=True     # 디버깅용 로그 출력
) 

# 프록시 설정 (필요 시 적절한 프록시 서버 구성)
proxy_config = {
    "server": "http://proxy.example.com:8080",
    "username": "user",
    "password": "pass"
}

proxy_config = base_config.clone(
    proxy_config=proxy_config,  
)

# 크롤러 실행 설정
run_config = CrawlerRunConfig()   

async def extract_by_url(url , browser_config: BrowserConfig, run_config: CrawlerRunConfig):
    """ 
    크롤링 실행 함수
    """

    async with AsyncWebCrawler(config=browser_config) as crawler:   # 크롤러 객체 생성
        result = await crawler.arun(
            url=url,
            config=run_config   # 크롤러 실행 설정
        )
        
    return result


# 크롤링 실행 및 결과 출력
result = asyncio.run(extract_by_url(etf_url, base_config, run_config))
# result = loop.run_until_complete(extract_by_url(etf_url, base_config, run_config))
print(result.success)  # 크롤링 성공 여부
print(result.markdown)  # 크롤링 결과 (정제된 markdown 출력)

RuntimeError: This event loop is already running

### 5) **CrawlerRunConfig 설정** 

- 단어 수 제한, 추출 전략, 캐시 설정 등 **크롤링 옵션** 관리

- **js_code**와 **wait_for** 옵션으로 동적 콘텐츠 처리 가능

- **screenshot**과 **rate_limiting** 기능으로 크롤링 과정 제어

- **기본 구조**:

    ```python
    class CrawlerRunConfig:
        def __init__(
            word_count_threshold=200,      # 컨텐츠 최소 단어 수
            extraction_strategy=None,      # 데이터 추출 전략
            cache_mode=None,               # 캐시 설정
            js_code=None,                  # 실행할 JS 코드
            wait_for=None,                 # 대기 조건
            screenshot=False,              # 스크린샷 캡처
            enable_rate_limiting=False     # 속도 제한 활성화
        )
    ```

`(1) 기본 추출 설정`

In [ ]:
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig, CacheMode

run_config = CrawlerRunConfig(
    cache_mode=CacheMode.ENABLED,   # 캐시 활성화
    word_count_threshold=200,       # 컨텐츠 블록의 최소 단어 수 기준 설정 (짧은 문단이나 항목이 많은 사이트는 기준값을 낮춰야 함)
)

# 크롤링 실행 및 결과 출력
result = asyncio.run(extract_by_url(etf_url, base_config, run_config))
# result = loop.run_until_complete(extract_by_url(etf_url, base_config, run_config))
print(result.success)  # 크롤링 성공 여부
print(result.markdown)  # 크롤링 결과 (정제된 markdown 출력)

Task exception was never retrieved
future: <Task finished name='Task-97' coro=<Connection.run() done, defined at c:\fun\002_etfbot\.venv\Lib\site-packages\playwright\_impl\_connection.py:305> exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\asyncio\tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "c:\fun\002_etfbot\.venv\Lib\site-packages\playwright\_impl\_connection.py", line 312, in run
    await self._transport.connect()
  File "c:\fun\002_etfbot\.venv\Lib\site-packages\playwright\_impl\_transport.py", line 133, in connect
    raise exc
  File "c:\fun\002_etfbot\.venv\Lib\site-packages\playwright\_impl\_transport.py", line 120, in connect
    self._proc = await asyncio.create_subprocess_exec(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\Python

NotImplementedError: 

`(2) 추출 전략`

- 참조 문서: https://docs.crawl4ai.com/core/content-selection/

- 대상 페이지: https://news.ycombinator.com/newest

- robots.txt: https://www.ycombinator.com/robots.txt

In [ ]:
news_url = "https://news.ycombinator.com/newest"
news_url

'https://news.ycombinator.com/newest'

In [ ]:
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig, CacheMode

run_config = CrawlerRunConfig(
    css_selector="main.content",         # 크롤링 대상 컨텐츠 선택자
    word_count_threshold=10,             # 컨텐츠 블록의 최소 단어 수 기준 설정
    excluded_tags=["nav", "footer"],     # 제외할 태그 설정
    exclude_external_links=True,         # 외부 링크 제외 설정
    exclude_social_media_links=True,     # 소셜 미디어 링크 제외 설정
    exclude_domains=["ads.com", "spammytrackers.net"],   # 제외할 도메인 설정
    exclude_external_images=True,         # 외부 이미지 제외 설정
    cache_mode=CacheMode.BYPASS           # 캐시 비활성화
)

# 크롤링 실행 및 결과 출력
result = asyncio.run(extract_by_url(news_url, base_config, run_config))
# result = loop.run_until_complete(extract_by_url(etf_url, base_config, run_config))
print(result.success)  # 크롤링 성공 여부
print(result.markdown)  # 크롤링 결과 (정제된 markdown 출력)

Task exception was never retrieved
future: <Task finished name='Task-105' coro=<Connection.run() done, defined at c:\fun\002_etfbot\.venv\Lib\site-packages\playwright\_impl\_connection.py:305> exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.12_3.12.2800.0_x64__qbz5n2kfra8p0\Lib\asyncio\tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "c:\fun\002_etfbot\.venv\Lib\site-packages\playwright\_impl\_connection.py", line 312, in run
    await self._transport.connect()
  File "c:\fun\002_etfbot\.venv\Lib\site-packages\playwright\_impl\_transport.py", line 133, in connect
    raise exc
  File "c:\fun\002_etfbot\.venv\Lib\site-packages\playwright\_impl\_transport.py", line 120, in connect
    self._proc = await asyncio.create_subprocess_exec(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\Pytho

NotImplementedError: 

In [19]:
# CSS 기반 추출 전략
from crawl4ai.extraction_strategy import JsonCssExtractionStrategy

news_schema = {
    "name": "HN Stories",
    "baseSelector": "tr.athing",
    "fields": [
        {
            "name": "title",
            "selector": "td.title span.titleline a:first-child",
            "type": "text"
        },
        {
            "name": "url",
            "selector": "td.title span.titleline a:first-child",
            "type": "attribute", 
            "attribute": "href"
        },
        {
            "name": "rank",
            "selector": "td.title span.rank",
            "type": "text",
            "transform": lambda x: int(x.strip("."))
        }
    ]
}

extraction_config = CrawlerRunConfig(
    extraction_strategy=JsonCssExtractionStrategy(news_schema),
    cache_mode=CacheMode.BYPASS  # 항상 최신 데이터 가져오기
)


# 크롤링 실행 및 결과 출력
result = asyncio.run(extract_by_url(news_url, base_config, extraction_config))
# result = loop.run_until_complete(extract_by_url(etf_url, base_config, run_config))

# 구조화된 데이터 (Optional)
if result.extracted_content:
    data = json.loads(result.extracted_content)
    print(data)

NameError: name 'CacheMode' is not defined

In [ ]:
eval(result.extracted_content)[0]

In [ ]:
# 네이버 뉴스

news_url = "https://n.news.naver.com/article/654/0000103393?cds=news_media_pc&type=breakingnews"
news_url

In [ ]:
# CSS 기반 추출 전략
from crawl4ai.extraction_strategy import JsonCssExtractionStrategy

news_schema = {
    "name": "NewsArticles",
    "baseSelector": "div.newsct",
    "fields": [
        {
            "name": "title",
            "selector": "div.media_end_head div.media_end_head_title h2.media_end_head_headline span",
            "type": "text"
        },
    ]
}

extraction_config = CrawlerRunConfig(
    extraction_strategy=JsonCssExtractionStrategy(news_schema),
    process_iframes=True,
    remove_overlay_elements=True
)


news_url = "https://n.news.naver.com/article/654/0000103393?cds=news_media_pc&type=breakingnews"

# 크롤링 실행 및 결과 출력
result = asyncio.run(extract_by_url(news_url, base_config, extraction_config))
# result = loop.run_until_complete(extract_by_url(etf_url, base_config, extraction_config))

# 구조화된 데이터 (Optional)
if result.extracted_content:
    data = json.loads(result.extracted_content)
    print(data)

In [ ]:
import os
import json
import asyncio
from pydantic import BaseModel, Field
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig, CacheMode
from crawl4ai.async_configs import LLMConfig 
from crawl4ai.extraction_strategy import LLMExtractionStrategy

# 데이터 모델 정의
class NaverNewsArticle(BaseModel):
    title: str = Field(description="기사 제목")
    published_date: str = Field(description="발행일")
    author: str = Field(description="기자 이름") 
    content: str = Field(description="기사 본문")


async def extract_naver_news(news_url):
    # LLM 추출 전략 설정
    strategy = LLMExtractionStrategy(
        llm_config=LLMConfig(
            provider="openai/gpt-4.1-mini",
            api_token=os.getenv("OPENAI_API_KEY"),
            ),
        schema=NaverNewsArticle.model_json_schema(),
        extraction_type="schema",
        instruction="""
        네이버 뉴스 기사에서 다음 정보를 추출하세요:
        - title: 기사 제목
        - published_date: 발행일시 
        - author: 기자 이름
        - content: 기사 본문
        """
    )

    config = CrawlerRunConfig(
        exclude_external_links=True,
        extraction_strategy=strategy,
        cache_mode=CacheMode.BYPASS
    )

    async with AsyncWebCrawler() as crawler:
        result = await crawler.arun(url=news_url, config=config)

        print(result.extracted_content)
        
        if result.extracted_content:
            article = json.loads(result.extracted_content)
            return article
        return None

# 실행
news_url = "https://n.news.naver.com/article/654/0000103393?cds=news_media_pc&type=breakingnews"
article = asyncio.run(extract_naver_news(news_url)) # asyncio.run() 사용
# article = loop.run_until_complete(extract_naver_news(news_url))  #  윈도우 기준


print(article)

In [ ]:
pprint(article[0]['content'])

---

## [실습] **ETF 상세정보 수집**

- 다음 ETF 상세 페이지에서 다음 항목을 수집하는 함수를 작성합니다. 
- 대상 필드:
    - '한글명', '영문명', '종목코드', '상장일', '펀드형태', '기초지수명', '추적배수', '자산운용사',
    - '지정참가회사(AP)', '총보수(%)', '회계기간', '과세유형', '분배금 지급일', '홈페이지',
    - '기초 시장', '기초 자산', '기본 정보', '투자유의사항'

- 모두 5개의 ETF 종목에 대한 정보를 수집하여 csv 파일로 저장합니다. 

- **Hint**:
    - crawl4ai 사용하여 각 ETF 상세 페이지의 HTML 소스코드를 추출 (종목코드: 5자리 숫자에 유의)
    - pandas.read_html() 함수를 사용하여 HTML 소스코드에서 테이블 데이터를 추출 